In [3]:
import pandas as pd

In [4]:
data = [[1, 201, '2024-03-01 10:00:00', 'app_open', 'S001', None], [2, 201, '2024-03-01 10:05:00', 'scroll', 'S001', 500], [3, 201, '2024-03-01 10:10:00', 'scroll', 'S001', 750], [4, 201, '2024-03-01 10:15:00', 'scroll', 'S001', 600], [5, 201, '2024-03-01 10:20:00', 'scroll', 'S001', 800], [6, 201, '2024-03-01 10:25:00', 'scroll', 'S001', 550], [7, 201, '2024-03-01 10:30:00', 'scroll', 'S001', 900], [8, 201, '2024-03-01 10:35:00', 'app_close', 'S001', None], [9, 202, '2024-03-01 11:00:00', 'app_open', 'S002', None], [10, 202, '2024-03-01 11:02:00', 'click', 'S002', None], [11, 202, '2024-03-01 11:05:00', 'scroll', 'S002', 400], [12, 202, '2024-03-01 11:08:00', 'click', 'S002', None], [13, 202, '2024-03-01 11:10:00', 'scroll', 'S002', 350], [14, 202, '2024-03-01 11:15:00', 'purchase', 'S002', 50], [15, 202, '2024-03-01 11:20:00', 'app_close', 'S002', None], [16, 203, '2024-03-01 12:00:00', 'app_open', 'S003', None], [17, 203, '2024-03-01 12:10:00', 'scroll', 'S003', 1000], [18, 203, '2024-03-01 12:20:00', 'scroll', 'S003', 1200], [19, 203, '2024-03-01 12:25:00', 'click', 'S003', None], [20, 203, '2024-03-01 12:30:00', 'scroll', 'S003', 800], [21, 203, '2024-03-01 12:40:00', 'scroll', 'S003', 900], [22, 203, '2024-03-01 12:50:00', 'scroll', 'S003', 1100], [23, 203, '2024-03-01 13:00:00', 'app_close', 'S003', None], [24, 204, '2024-03-01 14:00:00', 'app_open', 'S004', None], [25, 204, '2024-03-01 14:05:00', 'scroll', 'S004', 600], [26, 204, '2024-03-01 14:08:00', 'scroll', 'S004', 700], [27, 204, '2024-03-01 14:10:00', 'click', 'S004', None], [28, 204, '2024-03-01 14:12:00', 'app_close', 'S004', None]]
app_events = pd.DataFrame(data, columns={
    "event_id": pd.Series(dtype="int"),
    "user_id": pd.Series(dtype="int"),
    "event_timestamp": pd.Series(dtype="datetime64[ns]"),
    "event_type": pd.Series(dtype="string"),   # varchar -> string
    "session_id": pd.Series(dtype="string"),   # varchar -> string
    "event_value": pd.Series(dtype="int")
})

In [11]:
app_events['if_scroll'] = (app_events['event_type'] == 'scroll').astype(int)
app_events['if_click'] = (app_events['event_type'] == 'click').astype(int)
app_events['if_purchase'] = (app_events['event_type'] == 'purchase').astype(int)

In [13]:
app_events.head(10)

,event_id,user_id,event_timestamp,event_type,session_id,event_value,if_scroll,if_click,if_purchase
0,1,201,2024-03-01 10:00:00,app_open,S001,NaN,0,0,0
1,2,201,2024-03-01 10:05:00,scroll,S001,500.0,1,0,0
2,3,201,2024-03-01 10:10:00,scroll,S001,750.0,1,0,0
3,4,201,2024-03-01 10:15:00,scroll,S001,600.0,1,0,0
4,5,201,2024-03-01 10:20:00,scroll,S001,800.0,1,0,0
5,6,201,2024-03-01 10:25:00,scroll,S001,550.0,1,0,0
6,7,201,2024-03-01 10:30:00,scroll,S001,900.0,1,0,0
7,8,201,2024-03-01 10:35:00,app_close,S001,NaN,0,0,0
8,9,202,2024-03-01 11:00:00,app_open,S002,NaN,0,0,0
9,10,202,2024-03-01 11:02:00,click,S002,NaN,0,1,0


In [17]:
res = app_events.groupby(['session_id', 'user_id']).agg(
    first_time = ('event_timestamp', 'first'),
    last_time = ('event_timestamp', 'last'),
    scroll_count = ('if_scroll', 'sum'),
    click_count=('if_click', 'sum'),
    purchase_count=('if_purchase', 'sum')
).reset_index()

In [26]:
res['session_duration_minutes'] = (pd.to_datetime(res['last_time']) - pd.to_datetime(res['first_time'])).dt.total_seconds() / 60

In [28]:
res['click_count_ratio'] = res['click_count'] / res['scroll_count']

In [32]:
res = res[(res['session_duration_minutes'] > 30) & (res['click_count_ratio'] < 0.2) & (res['purchase_count'] == 0)]
res[['session_id', 'user_id', 'session_duration_minutes', 'scroll_count']].sort_values(['scroll_count', 'session_id'], ascending=[0,1])

,session_id,user_id,session_duration_minutes,scroll_count
0,S001,201,35.0,6


In [22]:
pd.to_datetime(res['last_time'])

0   2024-03-01 10:35:00
1   2024-03-01 11:20:00
2   2024-03-01 13:00:00
3   2024-03-01 14:12:00
Name: last_time, dtype: datetime64[ns]

In [33]:
data = [[1, 101, 'Python Basics', '2024-01-05', 5], [1, 102, 'SQL Fundamentals', '2024-02-10', 4], [1, 103, 'JavaScript', '2024-03-15', 5], [1, 104, 'React Basics', '2024-04-20', 4], [1, 105, 'Node.js', '2024-05-25', 5], [1, 106, 'Docker', '2024-06-30', 4], [2, 101, 'Python Basics', '2024-01-08', 4], [2, 104, 'React Basics', '2024-02-14', 5], [2, 105, 'Node.js', '2024-03-20', 4], [2, 106, 'Docker', '2024-04-25', 5], [2, 107, 'AWS Fundamentals', '2024-05-30', 4], [3, 101, 'Python Basics', '2024-01-10', 3], [3, 102, 'SQL Fundamentals', '2024-02-12', 3], [3, 103, 'JavaScript', '2024-03-18', 3], [3, 104, 'React Basics', '2024-04-22', 2], [3, 105, 'Node.js', '2024-05-28', 3], [4, 101, 'Python Basics', '2024-01-12', 5], [4, 108, 'Data Science', '2024-02-16', 5], [4, 109, 'Machine Learning', '2024-03-22', 5]]
course_completions = pd.DataFrame(data, columns={
    "user_id": pd.Series(dtype="int"),
    "course_id": pd.Series(dtype="int"),
    "course_name": pd.Series(dtype="string"),           # corresponds to SQL VARCHAR
    "completion_date": pd.Series(dtype="datetime64[ns]"),  # corresponds to SQL DATE
    "course_rating": pd.Series(dtype="Int64")           # corresponds to SQL INT (nullable)
})

In [59]:
top_students = course_completions.groupby('user_id').agg(
    course_count=('course_id', 'count'),
    avg_rating=('course_rating', 'mean')
).reset_index()

In [61]:
top_students = top_students[(top_students['avg_rating']>= 4) & (top_students['course_count'] >=5)]
top_students = top_students['user_id'].to_list()
filtered_courses = course_completions[course_completions['user_id'].isin(top_students)].sort_values(['user_id','completion_date'])

In [62]:
filtered_courses

,user_id,course_id,course_name,completion_date,course_rating
0,1,101,Python Basics,2024-01-05,5
1,1,102,SQL Fundamentals,2024-02-10,4
2,1,103,JavaScript,2024-03-15,5
3,1,104,React Basics,2024-04-20,4
4,1,105,Node.js,2024-05-25,5
5,1,106,Docker,2024-06-30,4
6,2,101,Python Basics,2024-01-08,4
7,2,104,React Basics,2024-02-14,5
8,2,105,Node.js,2024-03-20,4
9,2,106,Docker,2024-04-25,5


In [63]:
from collections import defaultdict
course_freq = defaultdict(int)
for (user_id), g in filtered_courses.groupby('user_id'):
    extract_freq(g['course_name'])
    # break


In [64]:
def extract_freq(s_course):
    courses = s_course.to_list()
    for i in range(len(courses)-1):
        course_freq[(courses[i], courses[i+1])] += 1

In [65]:
res = []
for course_pair, count in course_freq.items():
    res.append({'first_course': course_pair[0], 'second_course': course_pair[1], 'transition_count': count})

In [68]:
result = pd.DataFrame(res).sort_values(['transition_count', 'first_course', 'second_course'], ascending=[0, 1, 1])

In [71]:
result['1st_lower'] = result['first_course'].map(lambda x: x.lower())
result['2nd_lower'] = result['second_course'].map(lambda x: x.lower())

In [73]:
result.sort_values(['transition_count', '1st_lower', '2nd_lower'], ascending=[0, 1, 1])[['first_course', 'second_course', 'transition_count']]

,first_course,second_course,transition_count
4,Node.js,Docker,2
3,React Basics,Node.js,2
6,Docker,AWS Fundamentals,1
2,JavaScript,React Basics,1
5,Python Basics,React Basics,1
0,Python Basics,SQL Fundamentals,1
1,SQL Fundamentals,JavaScript,1


In [108]:
data = [[1, '2024-01-01', 'login'], [1, '2024-01-02', 'login'], [1, '2024-01-03', 'login'], [1, '2024-01-04', 'login'], [1, '2024-01-05', 'login'], [1, '2024-01-06', 'logout'], [2, '2024-01-01', 'click'], [2, '2024-01-02', 'click'], [2, '2024-01-03', 'click'], [2, '2024-01-04', 'click'], [3, '2024-01-01', 'view'], [3, '2024-01-02', 'view'], [3, '2024-01-03', 'view'], [3, '2024-01-04', 'view'], [3, '2024-01-05', 'view'], [3, '2024-01-06', 'view'], [3, '2024-01-07', 'view']]
activity = pd.DataFrame(data, columns={
    "user_id": pd.Series(dtype="int"),
    "action_date": pd.Series(dtype="datetime64[ns]"),
    "action": pd.Series(dtype="string")
})

In [ ]:
activity['min_date'] = activity.action_date.min()
activity['date_diff'] = (pd.to_datetime(activity['action_date']) - pd.to_datetime(activity['min_date'])).dt.days
activity['day_count'] = activity.groupby(['user_id', 'action_date'])['action'].transform('count')
activity = activity[activity['day_count'] == 1]
activity['streak'] = activity.date_diff - activity.index

In [114]:
activity

,user_id,action_date,action,min_date,date_diff,day_count,streak
0,1,2024-01-01,login,2024-01-01,0,1,0
1,1,2024-01-02,login,2024-01-01,1,1,0
2,1,2024-01-03,login,2024-01-01,2,1,0
3,1,2024-01-04,login,2024-01-01,3,1,0
4,1,2024-01-05,login,2024-01-01,4,1,0
5,1,2024-01-06,logout,2024-01-01,5,1,0
6,2,2024-01-01,click,2024-01-01,0,1,-6
7,2,2024-01-02,click,2024-01-01,1,1,-6
8,2,2024-01-03,click,2024-01-01,2,1,-6
9,2,2024-01-04,click,2024-01-01,3,1,-6


In [122]:
res = activity.groupby(['user_id', 'action', 'streak']).agg(
    streak_length=('action_date', 'count'),
    start_date=('action_date', 'min'),
    end_date =('action_date', 'max')
).reset_index()

In [124]:
res[res['streak_length'] >=5].sort_values(['streak_length', 'user_id'], ascending=[0, 1]).drop_duplicates('user_id').drop('streak', axis=1)

,user_id,action,streak_length,start_date,end_date
3,3,view,7,2024-01-01,2024-01-07
0,1,login,5,2024-01-01,2024-01-05


In [177]:
data = [[1, 'Alice Smith', 28], [2, 'Bob Johnson', 35], [3, 'Carol Davis', 42], [4, 'David Wilson', 31], [5, 'Emma Brown', 29]]
patients = pd.DataFrame(data, columns={
    'patient_id': pd.Series(dtype='int'),
    'patient_name': pd.Series(dtype='str'),
    'age': pd.Series(dtype='int')
})
data = [[1, 1, '2023-01-15', 'Positive'], [2, 1, '2023-01-25', 'Negative'], [3, 2, '2023-02-01', 'Positive'], [4, 2, '2023-02-05', 'Inconclusive'], [5, 2, '2023-02-12', 'Negative'], [6, 3, '2023-01-20', 'Negative'], [7, 3, '2023-02-10', 'Positive'], [8, 3, '2023-02-20', 'Negative'], [9, 4, '2023-01-10', 'Positive'], [10, 4, '2023-01-18', 'Positive'], [11, 5, '2023-02-15', 'Negative'], [12, 5, '2023-02-20', 'Negative']]
covid_tests = pd.DataFrame(data, columns={
    'test_id': pd.Series(dtype='int'),
    'patient_id': pd.Series(dtype='int'),
    'test_date': pd.Series(dtype='datetime64[ns]'),
    'result': pd.Series(dtype='str')
})

In [178]:
patients

,patient_id,patient_name,age
0,1,Alice Smith,28
1,2,Bob Johnson,35
2,3,Carol Davis,42
3,4,David Wilson,31
4,5,Emma Brown,29


In [179]:
covid_tests['test_date'] = pd.to_datetime(covid_tests['test_date'])

In [183]:
tmp1 = covid_tests[covid_tests['result'] == 'Positive'].groupby('patient_id')['test_date'].min().reset_index().rename(columns={'test_date': 'first_positive'})

In [184]:
tmp1

,patient_id,first_positive
0,1,2023-01-15
1,2,2023-02-01
2,3,2023-02-10
3,4,2023-01-10


In [185]:
tmp2 = covid_tests.merge(tmp1, on='patient_id', how='left')

In [186]:
tmp2

,test_id,patient_id,test_date,result,first_positive
0,1,1,2023-01-15,Positive,2023-01-15
1,2,1,2023-01-25,Negative,2023-01-15
2,3,2,2023-02-01,Positive,2023-02-01
3,4,2,2023-02-05,Inconclusive,2023-02-01
4,5,2,2023-02-12,Negative,2023-02-01
5,6,3,2023-01-20,Negative,2023-02-10
6,7,3,2023-02-10,Positive,2023-02-10
7,8,3,2023-02-20,Negative,2023-02-10
8,9,4,2023-01-10,Positive,2023-01-10
9,10,4,2023-01-18,Positive,2023-01-10


In [159]:
tmp3 = tmp2[tmp2['first_positive'].notna() & (tmp2['test_date'] >= tmp2['first_positive']) & (tmp2['result'] == 'Negative')].groupby('patient_id').agg(
    first_negative=('test_date', 'first')
).reset_index()

In [166]:
tmp1

,patient_id,first_positive
0,1,2023-01-15
2,2,2023-02-01
6,3,2023-02-10
8,4,2023-01-10
9,4,2023-01-10


In [168]:
tmp4 = tmp3.merge(tmp1, on='patient_id', how='left')

In [170]:
tmp4['recovery_time'] = (tmp4['first_negative'] - tmp4['first_positive']).dt.days

In [173]:
patients.merge(tmp4[['patient_id', 'recovery_time']], on='patient_id', how='right').sort_values(['recovery_time', 'patient_name'])

,patient_id,patient_name,age,recovery_time
0,1,Alice Smith,28,10
2,3,Carol Davis,42,10
1,2,Bob Johnson,35,11


In [219]:
data = [[1, 'Alice Johnson'], [2, 'Bob Smith'], [3, 'Carol Davis'], [4, 'David Wilson'], [5, 'Emma Brown']]
drivers = pd.DataFrame(data, columns={
    'driver_id': pd.Series(dtype='int'),
    'driver_name': pd.Series(dtype='str')
})
data = [[1, 1, '2023-02-15', 120.5, 10.2], [2, 1, '2023-03-20', 200.0, 16.5], [3, 1, '2023-08-10', 150.0, 11.0], [4, 1, '2023-09-25', 180.0, 12.5], [5, 2, '2023-01-10', 100.0, 9.0], [6, 2, '2023-04-15', 250.0, 22.0], [7, 2, '2023-10-05', 200.0, 15.0], [8, 3, '2023-03-12', 80.0, 8.5], [9, 3, '2023-05-18', 90.0, 9.2], [10, 4, '2023-07-22', 160.0, 12.8], [11, 4, '2023-11-30', 140.0, 11.0], [12, 5, '2023-02-28', 110.0, 11.5]]
trips = pd.DataFrame(data, columns={
    'trip_id': pd.Series(dtype='int'),
    'driver_id': pd.Series(dtype='int'),
    'trip_date': pd.Series(dtype='datetime64[ns]'),
    'distance_km': pd.Series(dtype='float'),
    'fuel_consumed': pd.Series(dtype='float')
})

In [220]:
drivers

,driver_id,driver_name
0,1,Alice Johnson
1,2,Bob Smith
2,3,Carol Davis
3,4,David Wilson
4,5,Emma Brown


In [221]:
trips['trip_date'] = pd.to_datetime(trips['trip_date'])

In [222]:
trips['month_date'] = trips['trip_date'].dt.strftime('%m-%d')

In [223]:
trips['fuel_efficiency'] = trips['distance_km'] / trips['fuel_consumed']

In [224]:
first_half = trips[trips['month_date'] <= '06-30'].groupby('driver_id')['fuel_efficiency'].mean().reset_index().rename(columns={'fuel_efficiency':'first_half_avg'})

In [225]:
second_half = trips[trips['month_date'] >= '07-01'].groupby('driver_id')['fuel_efficiency'].mean().reset_index().rename(columns={'fuel_efficiency':'second_half_avg'})

In [226]:
first_half

,driver_id,first_half_avg
0,1,11.967469
1,2,11.237374
2,3,9.597187
3,5,9.565217


In [227]:
second_half

,driver_id,second_half_avg
0,1,14.018182
1,2,13.333333
2,4,12.613636


In [229]:
res = drivers.merge(first_half).merge(second_half)

In [230]:
res

,driver_id,driver_name,first_half_avg,second_half_avg
0,1,Alice Johnson,11.967469,14.018182
1,2,Bob Smith,11.237374,13.333333


In [231]:
res['first_half_avg'] = round(res['first_half_avg'], 2)
res['second_half_avg'] = round(res['second_half_avg'], 2)

In [233]:
res['efficiency_improvement'] = res['second_half_avg'] - res['first_half_avg']

In [235]:
res[res['efficiency_improvement'] > 0]

,driver_id,driver_name,first_half_avg,second_half_avg,efficiency_improvement
0,1,Alice Johnson,11.97,14.02,2.05
1,2,Bob Smith,11.24,13.33,2.09


In [236]:
data = [[1, 'Downtown Tech', 'New York'], [2, 'Suburb Mall', 'Chicago'], [3, 'City Center', 'Los Angeles'], [4, 'Corner Shop', 'Miami'], [5, 'Plaza Store', 'Seattle']]
stores = pd.DataFrame(data, columns=['store_id', 'store_name', 'location']).astype({'store_id': 'int64', 'store_name': 'string', 'location': 'string'})

data = [[1, 1, 'Laptop', 5, 999.99], [2, 1, 'Mouse', 50, 19.99], [3, 1, 'Keyboard', 25, 79.99], [4, 1, 'Monitor', 15, 299.99], [5, 2, 'Phone', 3, 699.99], [6, 2, 'Charger', 100, 25.99], [7, 2, 'Case', 75, 15.99], [8, 2, 'Headphones', 20, 149.99], [9, 3, 'Tablet', 2, 499.99], [10, 3, 'Stylus', 80, 29.99], [11, 3, 'Cover', 60, 39.99], [12, 4, 'Watch', 10, 299.99], [13, 4, 'Band', 25, 49.99], [14, 5, 'Camera', 8, 599.99], [15, 5, 'Lens', 12, 199.99]]
inventory = pd.DataFrame(data, columns=['inventory_id', 'store_id', 'product_name', 'quantity', 'price']).astype({'inventory_id': 'int64', 'store_id': 'int64', 'product_name': 'string', 'quantity': 'int64', 'price': 'float64'})


In [240]:
tmp1 = inventory.groupby('store_id').agg(
    low_price=('price', 'min'),
    high_price=('price', 'max'),
    product_count=('product_name', 'count')
).reset_index()

In [242]:
tmp2 = inventory.merge(tmp1, on='store_id')

In [257]:
low_price = tmp2[tmp2['price'] == tmp2['low_price']][['store_id', 'product_name', 'quantity']]
high_price = tmp2[tmp2['price'] == tmp2['high_price']][['store_id', 'product_name', 'quantity']]

In [258]:
low_price

,store_id,product_name,quantity
1,1,Mouse,50
6,2,Case,75
9,3,Stylus,80
12,4,Band,25
14,5,Lens,12


In [259]:
res = tmp1[['store_id','product_count']].merge(low_price).rename(columns={'product_name':'cheapest_product', 'quantity':'low_quantity'})\
    .merge(high_price).rename(columns={'product_name': 'most_exp_product', 'quantity':'high_quantity'})

In [264]:
res = res[(res['product_count'] >=3) & (res['low_quantity'] > res['high_quantity'])]

In [265]:
res['imbalance_ratio'] = res['low_quantity'] / res['high_quantity']

In [266]:
res

,store_id,product_count,cheapest_product,low_quantity,most_exp_product,high_quantity,imbalance_ratio
0,1,4,Mouse,50,Laptop,5,10.0
1,2,4,Case,75,Phone,3,25.0
2,3,3,Stylus,80,Tablet,2,40.0


In [268]:
stores.merge(res, how='right')[['store_id', 'store_name', 'location', 'most_exp_product', 'cheapest_product', 'imbalance_ratio']]

,store_id,store_name,location,most_exp_product,cheapest_product,imbalance_ratio
0,1,Downtown Tech,New York,Laptop,Mouse,10.0
1,2,Suburb Mall,Chicago,Phone,Case,25.0
2,3,City Center,Los Angeles,Tablet,Stylus,40.0


In [321]:
data = [[1, 101, '2024-03-01 12:30:00', 25.5, 'card', 5], [2, 101, '2024-03-02 19:15:00', 32.0, 'app', 4], [3, 101, '2024-03-03 13:45:00', 28.75, 'card', 5], [4, 101, '2024-03-04 20:30:00', 41.0, 'app', None], [5, 102, '2024-03-01 11:30:00', 18.5, 'cash', 4], [6, 102, '2024-03-02 12:00:00', 22.0, 'card', 3], [7, 102, '2024-03-03 15:30:00', 19.75, 'cash', None], [8, 103, '2024-03-01 19:00:00', 55.0, 'app', 5], [9, 103, '2024-03-02 20:45:00', 48.5, 'app', 4], [10, 103, '2024-03-03 18:30:00', 62.0, 'card', 5], [11, 104, '2024-03-01 10:00:00', 15.0, 'cash', 3], [12, 104, '2024-03-02 09:30:00', 18.0, 'cash', 2], [13, 104, '2024-03-03 16:00:00', 20.0, 'card', 3], [14, 105, '2024-03-01 12:15:00', 30.0, 'app', 4], [15, 105, '2024-03-02 13:00:00', 35.5, 'app', 5], [16, 105, '2024-03-03 11:45:00', 28.0, 'card', 4]]
restaurant_orders = pd.DataFrame(data, columns={
    "order_id": pd.Series(dtype="int"),
    "customer_id": pd.Series(dtype="int"),
    "order_timestamp": pd.Series(dtype="datetime64[ns]"),
    "order_amount": pd.Series(dtype="float"),
    "payment_method": pd.Series(dtype="string"),
    "order_rating": pd.Series(dtype="Int64")  # nullable integer for ratings that can be NULL
})

In [322]:
restaurant_orders['order_timestamp'] = pd.to_datetime(restaurant_orders['order_timestamp'])

In [323]:
def check_peak_hour(s):
    start1 = pd.to_datetime('11:00:00').time()
    end1 = pd.Timestamp('14:00:00').time()
    start2 = pd.Timestamp('18:00:00').time()
    end2 = pd.Timestamp('21:00:00').time()
    
    if start1 <= s.time() <= end1:
        return True
    elif start2 <= s.time() <= end2:
        return True
    return False
    
restaurant_orders['peak_hours'] = restaurant_orders['order_timestamp'].map(check_peak_hour)

In [324]:
restaurant_orders['rated'] = restaurant_orders['order_rating'].notna()

In [325]:
res = restaurant_orders.groupby('customer_id').agg(
    total_orders = ('order_id', 'count'),
    average_rating = ('order_rating', 'mean'),
    peak_ratio = ('peak_hours', 'mean'),
    rate_ratio = ('rated', 'mean')
).reset_index()

In [326]:
res['average_rating'] = res['average_rating'].round(2)

In [327]:
res['peak_hour_percentage'] = (res['peak_ratio'] * 100).round()

In [328]:
res = res[(res['total_orders'] >=3) & (res['peak_hour_percentage'] >=60) & (res['average_rating'] >=4) & (res['rate_ratio'] >=.5)]

In [330]:
res.drop(['peak_ratio', 'rate_ratio'], axis=1).sort_values(['average_rating', 'customer_id'], ascending=False)

,customer_id,total_orders,average_rating,peak_hour_percentage
2,103,3,4.67,100.0
0,101,4,4.67,100.0
4,105,3,4.33,100.0


In [287]:
pd.Timestamp('11:00:00').time()

datetime.time(11, 0)

In [288]:
pd.Timestamp.today().time()

datetime.time(20, 58, 43, 436077)

In [331]:
data = [[1, 501, '2024-01-01', 'start', 'premium', 29.99], [2, 501, '2024-02-15', 'downgrade', 'standard', 19.99], [3, 501, '2024-03-20', 'downgrade', 'basic', 9.99], [4, 502, '2024-01-05', 'start', 'standard', 19.99], [5, 502, '2024-02-10', 'upgrade', 'premium', 29.99], [6, 502, '2024-03-15', 'downgrade', 'basic', 9.99], [7, 503, '2024-01-10', 'start', 'basic', 9.99], [8, 503, '2024-02-20', 'upgrade', 'standard', 19.99], [9, 503, '2024-03-25', 'upgrade', 'premium', 29.99], [10, 504, '2024-01-15', 'start', 'premium', 29.99], [11, 504, '2024-03-01', 'downgrade', 'standard', 19.99], [12, 504, '2024-03-30', 'cancel', None, 0.0], [13, 505, '2024-02-01', 'start', 'basic', 9.99], [14, 505, '2024-02-28', 'upgrade', 'standard', 19.99], [15, 506, '2024-01-20', 'start', 'premium', 29.99], [16, 506, '2024-03-10', 'downgrade', 'basic', 9.99]]
subscription_events = pd.DataFrame(data, columns={
    "event_id": pd.Series(dtype="int"),
    "user_id": pd.Series(dtype="int"),
    "event_date": pd.Series(dtype="datetime64[ns]"),  # corresponds to SQL DATE
    "event_type": pd.Series(dtype="string"),
    "plan_name": pd.Series(dtype="string"),           # can be NULL for cancel events
    "monthly_amount": pd.Series(dtype="float")        # corresponds to DECIMAL(10,2)
})

In [345]:
subscription_events['event_date'] = pd.to_datetime(subscription_events['event_date'])

In [346]:
def check_downgrade(s):
    if s=='downgrade':
        return True
    return False
subscription_events['downgraded'] = subscription_events['event_type'].map(check_downgrade)

In [347]:
subscription_events

,event_id,user_id,event_date,event_type,plan_name,monthly_amount,downgraded
0,1,501,2024-01-01,start,premium,29.99,False
1,2,501,2024-02-15,downgrade,standard,19.99,True
2,3,501,2024-03-20,downgrade,basic,9.99,True
3,4,502,2024-01-05,start,standard,19.99,False
4,5,502,2024-02-10,upgrade,premium,29.99,False
5,6,502,2024-03-15,downgrade,basic,9.99,True
6,7,503,2024-01-10,start,basic,9.99,False
7,8,503,2024-02-20,upgrade,standard,19.99,False
8,9,503,2024-03-25,upgrade,premium,29.99,False
9,10,504,2024-01-15,start,premium,29.99,False


In [351]:
res = subscription_events.sort_values(['user_id', 'event_date']).groupby(['user_id']).agg(
    last_event=('event_type', 'last'),
    current_monthly_amount=('monthly_amount', 'last'),
    current_plan=('plan_name', 'last'),
    max_historical_amount=('monthly_amount', 'max'),
    downgrade_count=('downgraded', 'sum'),
    first_date=('event_date', 'min'),
    last_date=('event_date', 'max')
).reset_index()

In [352]:
res['days_as_subscriber'] = (res['last_date'] - res['first_date']).dt.days

In [358]:
res[(res['last_event'] != 'cancel') & 
    (res['downgrade_count'] > 0) & 
    (res['current_monthly_amount'] *2 < res['max_historical_amount']) & 
    (res['days_as_subscriber'] > 60)][['user_id', 'current_plan', 'current_monthly_amount', 'max_historical_amount', 'days_as_subscriber']]\
    .sort_values(['days_as_subscriber', 'user_id'], ascending=[0,1])

,user_id,current_plan,current_monthly_amount,max_historical_amount,days_as_subscriber
0,501,basic,9.99,29.99,79
1,502,basic,9.99,29.99,70


In [360]:
data = [[1, 101, 'like'], [1, 102, 'like'], [1, 103, 'like'], [1, 104, 'wow'], [1, 105, 'like'], [2, 201, 'like'], [2, 202, 'wow'], [2, 203, 'sad'], [2, 204, 'like'], [2, 205, 'wow'], [3, 301, 'love'], [3, 302, 'love'], [3, 303, 'love'], [3, 304, 'love'], [3, 305, 'love']]
reactions = pd.DataFrame(data, columns={
    "user_id": pd.Series(dtype="int"),
    "content_id": pd.Series(dtype="int"),
    "reaction": pd.Series(dtype="string")
})

In [367]:
def dominate_reaction(s):
    from collections import Counter
    counts = Counter(s)
    dominate_reaction = None
    max_value = 0
    for key, value in counts.items():
        if value > max_value:
            dominate_reaction = key
            max_value = value
    return dominate_reaction
reactions.groupby('user_id')['reaction'].apply(dominate_reaction).reset_index()

,user_id,reaction
0,1,like
1,2,like
2,3,love
